# Text processing

In [2]:
import re
import spacy
import torch
from sentence_transformers import SentenceTransformer
from transformers import pipeline, BartForConditionalGeneration, BartTokenizer

# Load SpaCy for NER
nlp = spacy.load("en_core_web_sm")

# Load the pre-trained BART model for text normalization
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")

# Load Sentence Transformer for embeddings
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Load emotion detection model
emotion_analyzer = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment")

# --- NEW DISFLUENCY CLEANER FUNCTION ---
def remove_disfluencies(text):
    """Generalized disfluency removal without hardcoding."""
    # Pattern: repeated 'um', 'uh', 'like', 'hmm', etc
    pattern = r'\b(um+|uh+|like+|hmm+|hmmm+|you know|i mean)\b'
    text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    
    # Remove multiple dots "..." -> "."
    text = re.sub(r'\.{2,}', '.', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# --- USE BART TO NORMALIZE ---
def clean_informal_speech(text):
    """Summarizes and normalizes messy text using BART."""
    inputs = bart_tokenizer(text, return_tensors="pt", max_length=1024, truncation=True, padding="longest")
    
    # BART generation parameters
    summary_ids = bart_model.generate(
        inputs["input_ids"],
        num_beams=4,
        min_length=20,
        max_length=200,
        length_penalty=2.0,
        early_stopping=True
    )
    
    cleaned_text = bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return cleaned_text

# --- FINAL DREAM PREPROCESSOR ---
def process_dream_description(text):
    """Pipeline: clean → normalize → extract entities → embed → emotion."""
    
    # Step 1: Light disfluency cleanup
    text_no_disfluencies = remove_disfluencies(text)
    
    # Step 2: Summarize/normalize messy text
    cleaned_text = clean_informal_speech(text_no_disfluencies)
    print(f"Cleaned Text:\n{cleaned_text}\n")  # Optional: print for debugging
    
    # Step 3: NER
    doc = nlp(cleaned_text)
    entities = {ent.label_: ent.text for ent in doc.ents}
    
    # Step 4: Embedding
    sentence_embedding = embedder.encode(cleaned_text)
    
    # Step 5: Emotion
    sentiment = emotion_analyzer(cleaned_text)[0]

    return {
        "entities": entities,
        "embedding": sentence_embedding.tolist(),
        "sentiment": sentiment,
        "cleaned_text": cleaned_text
    }



Device set to use mps:0


In [3]:
import pandas as pd

# Specify the path to your CSV file
csv_path = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/Reports_Filtered.csv"

# Load the CSV file
df = pd.read_csv(csv_path)

# Check if the column exists
if "Text of Report" not in df.columns:
    raise ValueError("Column 'Text of Report' not found in CSV!")

# Get the first dream report
first_text = df.loc[0, "Text of Report"]
print(f"Processing first entry:\n{first_text}\n")

# Apply the processing function to the first entry
first_output = process_dream_description(first_text)

# Print the structured output for tuning
import json
print(json.dumps(first_output, indent=4))


Processing first entry:
I umm I was with my parents and we're having a quarrel over the best… And we're just I remember having a really hard figuring out where we should go. And uh, umm...umm...ummm...I'm just having a long, difficult discussion with my parents, and eating is part of it, but umm...I...its it lasted forever it's like a really long discussion.... umm I can't remember.. It was...I think it was about someone. it was about... it was also about stuff going on with people. It wasn't too.. It was kind of serious... ummm... that's it. 

Cleaned Text:
I I was with my parents and we're having a quarrel over the best… And we're just I remember having a really hard figuring out where we should go. And , .I'm just having a long, difficult discussion.

{
    "entities": {},
    "embedding": [
        0.10152437537908554,
        -0.009022869169712067,
        -0.005700125824660063,
        -0.013585071079432964,
        -0.0299137681722641,
        0.0029442845843732357,
        -0.0

# Flan T5 Base test

In [6]:
import pandas as pd
import spacy 
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import re

# Load SpaCy model for NER
nlp = spacy.load("en_core_web_sm")

# Load Flan-T5 for cleaning informal speech
flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def clean_informal_speech(text):
    prompt = f"Paraphrase and clean this dream description: {text}"
    inputs = flan_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = flan_model.generate(**inputs, max_length=512, num_beams=4, length_penalty=1.5)
    cleaned_text = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return cleaned_text

def light_polish(text):
    """Extra polish after Flan cleaning: removes stutters, dangling periods, etc."""
    text = re.sub(r"\b(uh+|um+|hmm+|like+)\b", "", text, flags=re.IGNORECASE)  # remove leftover filler
    text = re.sub(r"\.\s*\.", ".", text)  # fix multiple periods
    text = re.sub(r"\s{2,}", " ", text)  # fix extra spaces
    text = text.strip()
    return text


def process_dream_description(text):
    """Pipeline that cleans text, extra polishes, extracts entities, actions, colors, and sentiment."""
    
    # Clean the informal speech
    cleaned_text = clean_informal_speech(text)
    cleaned_text = light_polish(cleaned_text)  # new light polish step
    
    print(f"Cleaned Text:\n{cleaned_text}\n")  # For tuning and checking
    
    # NER extraction
    doc = nlp(cleaned_text)
    entities = {ent.label_: ent.text for ent in doc.ents}
    
    # Sentence embeddings
    embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    sentence_embedding = embedder.encode(cleaned_text)
    
    # Emotion detection
    emotion_analyzer = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment")
    sentiment = emotion_analyzer(cleaned_text)[0]

    return {
        "entities": entities,
        "embedding": sentence_embedding.tolist(),
        "sentiment": sentiment,
        "cleaned_text": cleaned_text
    }


In [7]:
import pandas as pd
import json

# Specify the path to your CSV file
csv_path = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/Reports_Filtered.csv"

# Load the CSV file
df = pd.read_csv(csv_path)

# Check if the column exists
if "Text of Report" not in df.columns:
    raise ValueError("Column 'Text of Report' not found in CSV!")

# Get the first dream report
first_text = df.loc[0, "Text of Report"]
print(f"Processing first entry:\n{first_text}\n")

# Apply the processing function to the first entry
first_output = process_dream_description(first_text)

# Print the structured output nicely
print(json.dumps(first_output, indent=4))


Processing first entry:
I umm I was with my parents and we're having a quarrel over the best… And we're just I remember having a really hard figuring out where we should go. And uh, umm...umm...ummm...I'm just having a long, difficult discussion with my parents, and eating is part of it, but umm...I...its it lasted forever it's like a really long discussion.... umm I can't remember.. It was...I think it was about someone. it was about... it was also about stuff going on with people. It wasn't too.. It was kind of serious... ummm... that's it. 

Cleaned Text:
I remember having a really hard figuring out where we should go. And , .....I'm just having a long, difficult discussion with my parents, and eating is part of it, but ..I can't remember. It was about someone. It was also about stuff going on with people. It wasn't too. It was kind of serious.



Device set to use mps:0


{
    "entities": {},
    "embedding": [
        0.01542880479246378,
        0.05516396835446358,
        -0.02669941447675228,
        0.022975319996476173,
        -0.02350660040974617,
        0.038308944553136826,
        0.019027356058359146,
        -0.024913180619478226,
        0.011275401338934898,
        -0.10994724184274673,
        0.07763252407312393,
        0.06316745281219482,
        -0.046938542276620865,
        0.004088917747139931,
        0.01643536426126957,
        -0.021489815786480904,
        0.059424590319395065,
        -0.046154603362083435,
        -0.008058859966695309,
        0.0037686906289309263,
        0.04481411725282669,
        0.0911208912730217,
        0.010614387691020966,
        0.038230545818805695,
        0.01364060677587986,
        0.061178188771009445,
        0.041433218866586685,
        -0.03243652731180191,
        -0.06922665983438492,
        0.030755385756492615,
        0.08295463770627975,
        -0.007720947731286287,
  

In [8]:
import pandas as pd
import spacy 
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import re
from tqdm import tqdm

# Load SpaCy model for NER
nlp = spacy.load("en_core_web_sm")

# Load Flan-T5 for cleaning informal speech
flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

# Embedder
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Emotion detector
emotion_analyzer = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment")

# === Cleaning functions ===

def clean_informal_speech(text):
    prompt = f"Paraphrase and clean this dream description: {text}"
    inputs = flan_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = flan_model.generate(**inputs, max_length=512, num_beams=4, length_penalty=1.5)
    cleaned_text = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return cleaned_text

def light_polish(text):
    text = re.sub(r"\b(uh+|um+|hmm+|like+)\b", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\.\s*\.", ".", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()

# === Load CSV and Prepare ===

csv_path = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/Reports_Filtered.csv"
df = pd.read_csv(csv_path)

if "Text of Report" not in df.columns:
    raise ValueError("Column 'Text of Report' not found in CSV!")

# Lists to store outputs
cleaned_texts = []
embeddings = []
sentiments = []

# Process each dream
print("Processing dreams...")
for text in tqdm(df["Text of Report"]):
    if pd.isna(text):
        cleaned_texts.append("")
        embeddings.append(torch.zeros(384))  # empty embedding (MiniLM output size is 384)
        sentiments.append({"label": "neutral", "score": 0.0})
        continue
    
    # Clean
    cleaned = clean_informal_speech(text)
    cleaned = light_polish(cleaned)
    cleaned_texts.append(cleaned)
    
    # Embed
    embedding = embedder.encode(cleaned)
    embeddings.append(torch.tensor(embedding))
    
    # Sentiment
    sentiment = emotion_analyzer(cleaned)[0]
    sentiments.append(sentiment)

# Add cleaned text and sentiment back to DataFrame
df["Cleaned Text"] = cleaned_texts
df["Sentiment Label"] = [s["label"] for s in sentiments]
df["Sentiment Score"] = [s["score"] for s in sentiments]

# Save updated CSV
updated_csv_path = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Reports_Filtered_Cleaned.csv"
df.to_csv(updated_csv_path, index=False)

# Save embeddings separately
embeddings_tensor = torch.stack(embeddings)
embeddings_save_path = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Reports_Embeddings.pt"
torch.save(embeddings_tensor, embeddings_save_path)

print(f"✅ All done! Cleaned CSV saved to {updated_csv_path}")
print(f"✅ Embeddings saved to {embeddings_save_path}")


Device set to use mps:0


Processing dreams...


100%|██████████| 238/238 [32:02<00:00,  8.08s/it]  

✅ All done! Cleaned CSV saved to /Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Reports_Filtered_Cleaned.csv
✅ Embeddings saved to /Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Reports_Embeddings.pt


# inspect the embeddings file

In [9]:
import torch

# Load the .pt file
embeddings = torch.load('/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/Reports_Embeddings.pt')

# Check what you loaded
print(type(embeddings))  # See the object type
print(len(embeddings))   # How many entries (should match the number of reports)

# If it's a tensor, check its shape
if isinstance(embeddings, torch.Tensor):
    print(embeddings.shape)
    # Shape should be (number_of_reports, embedding_dimension)

# If it's a list, print a few elements
elif isinstance(embeddings, list):
    for i in range(3):
        print(f"Embedding {i}: {embeddings[i]}")
        print(f"Length of embedding {i}: {len(embeddings[i])}")


<class 'torch.Tensor'>
238
torch.Size([238, 384])


# double checking if embedding match reports cases

In [12]:
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer  # or the model you used

# Load your CSV
csv_path = '/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/Reports_Filtered_Cleaned.csv'
df = pd.read_csv(csv_path)

# Load your saved embeddings
embeddings = torch.load('/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/Reports_Embeddings.pt')

# Load the same model you used originally
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Pick a random report (say, report 10)
i = 10
text = df.loc[i, 'Cleaned Text']  # replace 'cleaned_report' with your column name

# Encode it again
embedding_new = model.encode(text, convert_to_tensor=True)

# Compare it to your saved embedding
embedding_saved = embeddings[i]

# Make sure both embeddings are on CPU
embedding_new = embedding_new.cpu()
embedding_saved = embedding_saved.cpu()

# Now calculate cosine similarity
cosine_similarity = torch.nn.functional.cosine_similarity(embedding_new, embedding_saved, dim=0)
print(f"Cosine Similarity for Report {i}: {cosine_similarity.item():.4f}")




Cosine Similarity for Report 10: 1.0000
